# 🛡️ Silver Guard — Indian SMS Scam Detection Model

> Fine-tuned MobileBERT for detecting SMS scams with TRAI DLT header awareness

---

### 🎯 Objective
Binary classification model that outputs a **threat score (0.0–1.0)** for SMS messages. Designed for Indian context with support for DLT sender ID analysis.

### 📊 Data Sources
| Source | Type | Count |
|--------|------|-------|
| [UCI SMS Spam](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset) | Kaggle | ~5,500 |
| [India SMS Spam](https://www.kaggle.com/datasets/junioralive/india-spam-sms-classification) | Kaggle | ~2,200 |
| Synthetic Templates | Generated | ~5,200 |
| Personal SMS | User Data | ~5,100 |

### 🔧 Tech Stack
- **Model**: `google/mobilebert-uncased` (24M params)
- **Training**: PyTorch + HuggingFace Transformers
- **Export**: ONNX (for `onnxruntime_flutter`)
- **Runtime**: Google Colab T4 GPU

### 📁 Pipeline
```
Cell 1: Install & Config → Cell 2: Kaggle Data → Cell 3: Synthetic Data → Cell 4: Combine & Tokenize
    ↓
Cell 5: Train MobileBERT → Cell 6: Export ONNX → Cell 7: Test & Download
```

## 📦 Cell 1 — Install Dependencies & Configuration

Installs required packages and sets up global configuration. Heavy imports (PyTorch, Transformers) are **deferred to Cell 4** to avoid Colab dependency conflicts.

| Package | Purpose |
|---------|---------|
| `transformers` | MobileBERT model & tokenizer |
| `accelerate` | HuggingFace training utilities |
| `kaggle` | Dataset download API |
| `scikit-learn` | Train/test split, metrics |
| `torchvision` | Reinstalled to fix Colab conflicts |

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1: INSTALL & IMPORT (basics only — heavy imports deferred to where used)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q transformers accelerate kaggle pandas scikit-learn
!pip install -q torchvision --force-reinstall --no-deps
!pip install -q torchvision

import os, json, random, re, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Config
MAX_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 4
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Base imports done.")

## 📥 Cell 2 — Download Kaggle Datasets

Fetches two SMS spam datasets via Kaggle API and normalizes them into a unified schema.

**Output Schema:**
```python
{"text": str, "label": int, "source": str}  # label: 0=ham, 1=scam
```

**Auto-detection:** Handles varying column names (`message`/`text`/`sms`, `label`/`class`/`spam`) across datasets.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2: DOWNLOAD BOTH KAGGLE DATASETS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def load_kaggle_creds():
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return
    json_candidates = [
        Path("/content/kaggle.json"),              # ← Colab root folder (upload here)
        Path("kaggle.json"),                        # current working dir
        Path.home() / ".kaggle" / "kaggle.json",   # ~/.kaggle/kaggle.json
    ]
    for path in json_candidates:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if "username" in data and "key" in data:
                os.environ["KAGGLE_USERNAME"] = data["username"]
                os.environ["KAGGLE_KEY"] = data["key"]
                print(f"Kaggle credentials loaded from: {path}")
                return
    # Try env files
    env_candidates = [Path(".env"), Path("kaggle.env")]
    for env_path in env_candidates:
        if env_path.exists():
            with open(env_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith("#") or "=" not in line:
                        continue
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip("\"").strip("'")
                    if key in ("KAGGLE_USERNAME", "KAGGLE_KEY") and value:
                        os.environ.setdefault(key, value)
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        raise RuntimeError(
            "Kaggle credentials not found.\n"
            "Upload your kaggle.json to the /content/ folder in Colab's file browser, "
            "or set KAGGLE_USERNAME and KAGGLE_KEY as environment variables."
        )

load_kaggle_creds()

# ── Dataset 1: UCI SMS Spam Collection ────────────────────────────────────────
uci_dir = DATA_DIR / "uci_sms"
if not uci_dir.exists():
    uci_dir.mkdir(parents=True)
    print("Downloading UCI SMS Spam Collection from Kaggle...")
    os.system(f"kaggle datasets download -d uciml/sms-spam-collection-dataset -p {uci_dir} --unzip")
    print("Done.")
else:
    print("UCI dataset already exists.")

# Parse UCI dataset
df_uci = pd.DataFrame()
for f in uci_dir.iterdir():
    if f.suffix == ".csv":
        raw = pd.read_csv(f, encoding="latin-1")
        cols = raw.columns.tolist()
        text_col  = cols[1] if len(cols) > 1 else cols[0]
        label_col = cols[0]
        df_uci = pd.DataFrame({
            "text":   raw[text_col].astype(str),
            "label":  raw[label_col].apply(lambda x: 1 if str(x).lower().strip() == "spam" else 0),
            "source": "uci"
        })
        break
print(f"UCI: {len(df_uci)} msgs | spam={df_uci['label'].sum()}, ham={len(df_uci)-df_uci['label'].sum()}")

# ── Dataset 2: India SMS Spam Classification ──────────────────────────────────
india_dir = DATA_DIR / "india_sms"
if not india_dir.exists():
    india_dir.mkdir(parents=True)
    print("Downloading India SMS Spam Classification from Kaggle...")
    os.system(f"kaggle datasets download -d junioralive/india-spam-sms-classification -p {india_dir} --unzip")
    print("Done.")
else:
    print("India SMS dataset already exists.")

df_india = pd.DataFrame()
for f in india_dir.iterdir():
    if f.suffix == ".csv":
        raw = pd.read_csv(f)
        print(f"India SMS file: {f.name} | Columns: {list(raw.columns)}")
        text_col, label_col = None, None
        for col in raw.columns:
            cl = col.lower()
            if cl in ("message", "text", "sms", "msg", "content"):
                text_col = col
            elif cl in ("label", "class", "category", "type", "spam"):
                label_col = col
        if text_col  is None: text_col  = raw.columns[0]
        if label_col is None: label_col = raw.columns[1] if len(raw.columns) > 1 else raw.columns[0]

        unique_labels = raw[label_col].unique()
        print(f"Labels found: {unique_labels}")
        label_map = {
            lbl: (1 if str(lbl).lower().strip() in ("spam", "scam", "1", "1.0", "fraud", "phishing") else 0)
            for lbl in unique_labels
        }
        df_india = pd.DataFrame({
            "text":   raw[text_col].astype(str),
            "label":  raw[label_col].map(label_map),
            "source": "kaggle_india"
        }).dropna(subset=["label"])
        df_india["label"] = df_india["label"].astype(int)
        break

if df_india.empty:
    print("Warning: India SMS dataset empty — will rely on UCI + synthetic data.")
else:
    print(f"India: {len(df_india)} msgs | spam={df_india['label'].sum()}, ham={len(df_india)-df_india['label'].sum()}")

print(f"\nTotal from Kaggle: {len(df_uci) + len(df_india)} messages")

## 🏭 Cell 3 — Generate Synthetic Indian Scam SMS

Creates ~5,200 synthetic messages covering Indian scam patterns with **TRAI DLT header simulation**.

### Scam Categories (11 types, ~100 templates)
| Category | Examples |
|----------|----------|
| Digital Arrest | CBI/Police impersonation, Aadhaar fraud claims |
| Bank Freeze | KYC expiry, account suspension threats |
| OTP Fraud | Refund pending, share OTP requests |
| Link Phishing | Fake rewards, government subsidies |
| Lottery/Prize | KBC, WhatsApp lucky draw |
| Job Scam | Work from home, Telegram tasks |
| Parcel/Courier | Customs duty, contraband claims |

### TRAI DLT Header System
```
Format: XX-YYYYYYZ   (e.g., JD-SBINOT, VM-URGENT)
                ↑ Entity Type Suffix
  G = Government    (IRCTC, UIDAI, EPFO)
  T = Transactional (Banks, OTPs)
  S = Service       (Delivery, Alerts)
  P = Promotional   (Marketing, Offers)
```

**Key Signal**: Scams use phone numbers (`+919876543210`) or gibberish headers (`VM-URGENT`), while legit SMS use registered DLT IDs (`JD-SBINOT`).

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3: GENERATE SYNTHETIC INDIAN SCAM + HAM DATA (WITH TRAI DLT HEADERS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Template Variables ────────────────────────────────────────────────────────
INDIAN_BANKS = ["SBI", "HDFC Bank", "ICICI Bank", "Axis Bank", "PNB", "Bank of Baroda",
                "Canara Bank", "Union Bank", "Kotak Mahindra Bank", "IndusInd Bank",
                "Yes Bank", "IDBI Bank", "Bank of India", "Central Bank of India"]
UPI_APPS = ["PhonePe", "Google Pay", "Paytm", "BHIM", "Amazon Pay", "Cred", "MobiKwik"]
GOVT_AGENCIES = ["CBI", "Mumbai Police Cyber Cell", "Delhi Police", "Customs Department",
                 "Income Tax Department", "TRAI", "RBI", "Narcotics Bureau",
                 "Enforcement Directorate", "Ministry of Home Affairs"]
COURIER_SERVICES = ["FedEx", "BlueDart", "DTDC", "Delhivery", "India Post", "Ecom Express"]
FAKE_LINKS = ["bit.ly/3xK9mP2", "tinyurl.com/yb8d7fgh", "rb.gy/4kn2q8",
              "cutt.ly/verify-now", "shorturl.at/bEIW9", "is.gd/pqR5tV",
              "sfrfraud-alert.in/verify", "secure-bankupd.com/kyc",
              "gov-refund.online/claim", "reward-zone.info/win",
              "kycupdate-online.in/verify", "account-secure.net/login",
              "india-customs-clearance.com/pay", "tax-refund-gov.in/claim"]
AMOUNTS = ["₹50,000", "₹1,00,000", "₹2,50,000", "₹5,00,000", "₹10,00,000",
           "₹15,000", "₹25,000", "₹75,000", "₹3,50,000", "₹8,00,000",
           "₹50 lakh", "₹25 lakh", "₹10 lakh", "₹5 lakh", "₹1 crore"]
PHONE_NUMBERS = [f"+91 {random.randint(70000,99999)}{random.randint(10000,99999)}" for _ in range(20)]
OTP_CODES = [str(random.randint(100000, 999999)) for _ in range(50)]
NAMES = ["Rajesh Kumar", "Priya Sharma", "Amit Patel", "Sunita Devi", "Vikram Singh",
         "Anjali Gupta", "Mohammed Farhan", "Deepika Nair", "Suresh Reddy", "Kavita Joshi"]

# ══════════════════════════════════════════════════════════════════════════════
# TRAI DLT SENDER ID SYSTEM
# ══════════════════════════════════════════════════════════════════════════════
# Format: XX-YYYYYYZ  (XX = telecom circle, YYYYYY = entity name, Z = type)
#   G = Government entity       (IRCTC, UIDAI, EPFO, Income Tax, etc.)
#   T = Transactional           (banks, UPI, OTP alerts)
#   S = Service implicit/OTP
#   P = Promotional             (marketing, offers, sales)
# Scams come from: raw phone numbers, random gibberish IDs, or spoofed headers
# Key signal: a message claiming to be from "CBI" but sender is a phone number = SCAM
#             a message claiming to be from "SBI" with sender JD-SBINOT = legitimate

# ── Legitimate DLT Headers (real patterns) ────────────────────────────────────
# Government entities (end with G)
LEGIT_GOVT_HEADERS = [
    "JD-IRCTCG", "DL-ABORGG", "BZ-UIDAIG", "MH-EPFORG", "JD-INCOTG",
    "DL-PASSGG", "BZ-COWING", "JD-DGLOCG", "DL-UMANGG", "MH-MHGOVG",
    "DL-GOVTNG", "BZ-CBDTSG", "JD-EFIBEG", "MH-MPRSHG", "DL-PMJDYG",
]
# Bank transactional (end with T)
LEGIT_BANK_HEADERS = {
    "SBI":              ["JD-SBINOT", "BZ-SABORТ", "AD-SBIOTP"],
    "HDFC Bank":        ["AD-HDFCBK", "BZ-HDFCOT", "JD-HDFCNT"],
    "ICICI Bank":       ["AD-ICICIB", "BZ-ICICOT", "JD-ICICNT"],
    "Axis Bank":        ["AD-AXISBK", "BZ-AXISOT", "JD-AXISNT"],
    "PNB":              ["AD-PNBSMS", "BZ-PNBOTP"],
    "Bank of Baroda":   ["AD-BABORТ", "BZ-BOBSMS"],
    "Canara Bank":      ["AD-CANBKT", "BZ-CANOTP"],
    "Union Bank":       ["AD-UNIONT", "BZ-UBIOTP"],
    "Kotak Mahindra Bank": ["AD-KOTAKT", "BZ-KMBSMS"],
    "IndusInd Bank":    ["AD-INDUST", "BZ-INDOTP"],
    "Yes Bank":         ["AD-YESBKT", "BZ-YESOTP"],
    "IDBI Bank":        ["AD-IDBIBK", "BZ-IDBIOT"],
    "Bank of India":    ["AD-BKOINT", "BZ-BOIOTP"],
    "Central Bank of India": ["AD-CENTBK", "BZ-CBIOTP"],
}
# UPI/fintech transactional (end with T)
LEGIT_UPI_HEADERS = {
    "PhonePe":    ["JD-PHONPE", "BZ-PPEUPI"],
    "Google Pay": ["JD-GPAYOT", "BZ-GOOGPT"],
    "Paytm":      ["JD-PAYTMT", "BZ-PTMOTP"],
    "BHIM":       ["JD-BHIMUP", "BZ-BHIMOT"],
    "Amazon Pay": ["JD-AMZNPT", "BZ-AZNOTP"],
    "Cred":       ["JD-CREDAP", "BZ-CREDOT"],
    "MobiKwik":   ["JD-MBKWKT", "BZ-MKWOTP"],
}
# E-commerce / delivery (end with T)
LEGIT_ECOM_HEADERS = [
    "JD-AMZNIN", "BZ-FLPKRT", "AD-MYNTRA", "JD-SWGGYT", "BZ-ZOMATO",
    "AD-BGBSKT", "JD-MESHOT", "BZ-FXDEXT", "AD-BLUDRT", "JD-DTDCEX",
    "BZ-DLHVRY", "AD-INDPST", "JD-ECOMXT",
]
# Telecom (end with T)
LEGIT_TELECOM_HEADERS = [
    "JD-JIOINF", "BZ-AIRTLT", "AD-ABORVI", "JD-BSNLOT", "BZ-JIOFBT",
]
# Promotional (end with P)
LEGIT_PROMO_HEADERS = [
    "DM-DOMINO", "BZ-NYKAAT", "AD-DCATHL", "JD-CREDAP", "BZ-APLOAP",
    "AD-BIGBZP", "JD-SWGGYP", "BZ-ZOMATP", "DM-FLIPKP", "AD-MYNTRP",
]
# Courier (end with T)
LEGIT_COURIER_HEADERS = {
    "FedEx":        ["BZ-FXDEXT", "AD-FEDEXT"],
    "BlueDart":     ["AD-BLUDRT", "BZ-BLDART"],
    "DTDC":         ["AD-DTDCEX", "JD-DTDCOT"],
    "Delhivery":    ["BZ-DLHVRY", "AD-DLVRYT"],
    "India Post":   ["AD-INDPST", "BZ-INDPOT"],
    "Ecom Express": ["JD-ECOMXT", "AD-ECMEXT"],
}

# ── Scam Sender IDs (what scammers actually use) ─────────────────────────────
# Scammers send from: raw phone numbers, random short strings, or spoofed IDs
# that do NOT follow proper DLT format
SCAM_HEADERS = (
    # Raw phone numbers (most common for scam SMS)
    [f"+91{random.randint(6000000000,9999999999)}" for _ in range(30)] +
    # Gibberish / non-DLT format sender IDs
    ["VM-URGENT", "HP-ALERTS", "XX-WINNER", "TA-LUCKYY", "ZZ-CLAIMM",
     "QQ-REWARD", "YY-OFFRRR", "PP-PRIZED", "RR-CASHBK", "SS-VERIFY",
     "TT-CUSTOM", "UU-TAXREF", "VV-KYCUPD", "WW-LOANOK", "AA-JOBHIR"] +
    # Spoofed headers that look close but are wrong (e.g. end with wrong suffix)
    ["JD-SBIFRД", "BZ-HDFCFR", "AD-RBIFRД", "DL-CBIFRД", "MH-TRAIFR",
     "XX-IRCTCF", "QQ-EPFOFR", "ZZ-AXISFR", "PP-ICICFR", "YY-PNBFRD"] +
    # Some scammers try to use legit-looking headers with wrong suffix
    ["JD-SBINOP", "AD-HDFCBP", "BZ-RBIALP", "DL-CBIINP", "MH-TAXRFP"]
)

def get_scam_header():
    """Scam messages come from phone numbers or random/spoofed sender IDs."""
    return random.choice(SCAM_HEADERS)

def get_ham_header(category, bank=None, upi=None, courier=None):
    """Legitimate messages come from proper DLT-registered sender IDs."""
    if category == "bank_transactional" and bank:
        bank_name = bank
        for bk, headers in LEGIT_BANK_HEADERS.items():
            if bk in bank_name or bank_name in bk:
                return random.choice(headers)
        return random.choice(list(LEGIT_BANK_HEADERS.values())[0])
    elif category == "upi_transactional" and upi:
        for app, headers in LEGIT_UPI_HEADERS.items():
            if app in upi or upi in app:
                return random.choice(headers)
        return random.choice(list(LEGIT_UPI_HEADERS.values())[0])
    elif category == "ecommerce":
        return random.choice(LEGIT_ECOM_HEADERS)
    elif category == "government_legitimate":
        return random.choice(LEGIT_GOVT_HEADERS)
    elif category == "telecom":
        return random.choice(LEGIT_TELECOM_HEADERS)
    elif category == "travel_booking":
        return random.choice(LEGIT_ECOM_HEADERS + LEGIT_GOVT_HEADERS)
    elif category == "general_promotional":
        return random.choice(LEGIT_PROMO_HEADERS)
    else:
        return random.choice(LEGIT_ECOM_HEADERS)

def format_with_header(header, message):
    """Combine header and message with [SEP] token for model input."""
    return f"{header} [SEP] {message}"

# ── SCAM Templates (11 categories, ~105 templates) ───────────────────────────
SCAM_TEMPLATES = {
    "digital_arrest": [
        "ALERT: This is {agency}. Your Aadhaar number is linked to a money laundering case. A digital arrest warrant has been issued. Call {phone} immediately to avoid arrest.",
        "URGENT: {agency} has detected suspicious transactions from your account worth {amount}. A non-bailable warrant is being processed. Contact {phone} within 2 hours.",
        "WARNING from {agency}: Your mobile number is involved in financial fraud case #CR/{otp}. Appear on video call for digital verification or face arrest. {phone}",
        "This is {agency} calling. Your Aadhaar {otp} is used in drug trafficking. Digital arrest proceedings initiated. Cooperate immediately or FIR will be filed. Call {phone}",
        "NOTICE: {agency} cybercrime unit. Your bank account is linked to terrorism funding case. Stay on call for digital arrest verification. Failure = immediate arrest.",
        "⚠️ {agency} Alert: An arrest warrant has been issued in your name under PMLA Act. Your Aadhaar and PAN are flagged. Contact immediately at {phone} to resolve.",
        "Your identity documents are linked to a hawala transaction of {amount}. This is {agency}. A case under PMLA has been registered. Call {phone} to avoid arrest.",
        "IMPORTANT: {agency} investigation reveals your SIM card was used in cybercrime. Digital arrest will be executed in 24 hrs. Call {phone} for clearance.",
        "This is Officer from {agency}. FIR No. {otp} registered against your Aadhaar. You must join video call for statement recording or you will be arrested at your residence.",
        "{agency} CYBERCRIME ALERT: Your PAN card is linked to black money transactions of {amount}. Surrender digitally or armed police will arrive. {phone}",
    ],
    "bank_freeze": [
        "Dear Customer, your {bank} account has been FROZEN due to suspicious activity. Update your KYC immediately at {link} to restore access.",
        "ALERT: Your {bank} debit card ending XX89 will be BLOCKED in 24 hours. Complete mandatory verification: {link}",
        "RBI Notice: Your {bank} savings account is under investigation. Temporary freeze applied. Submit documents at {link} or account will be permanently closed.",
        "{bank} Security Alert: Unauthorized login detected on your net banking. Account temporarily suspended. Verify identity: {link}",
        "Your {bank} account KYC has expired. Services will be discontinued within 24 hrs. Update now: {link}. Sincerely, {bank} Team.",
        "URGENT: {bank} credit card payment of {amount} failed. Card will be reported to CIBIL. Clear dues immediately at {link}",
        "⚠️ {bank}: Your PAN is not linked. As per RBI circular, your account will be frozen on 15th. Link PAN now: {link}",
        "Dear Customer your {bank} a/c is temporarily blocked due to incomplete Aadhaar verification. Click {link} to complete or a/c will be closed.",
        "IMPORTANT: {bank} has detected your account being used for fraudulent transactions. Immediate KYC required: {link}. Ignore = account closure + CIBIL report.",
        "{bank} ALERT: Your fixed deposit of {amount} is at risk due to KYC non-compliance. Verify urgently at {link}",
    ],
    "otp_fraud": [
        "Your {bank} refund of {amount} is pending. Share OTP {otp} with our executive to process. Do NOT share with anyone else.",
        "Dear {name}, you have received a cashback of {amount} on {upi}. Confirm by sharing OTP {otp} on call.",
        "{upi} Alert: A payment of {amount} has been initiated from your account. If not done by you, share OTP {otp} to cancel the transaction.",
        "Your KYC verification OTP for {bank} is {otp}. Share this code with the bank executive on call to complete verification.",
        "Transaction of {amount} debited from your {bank} a/c. If unauthorized, call {phone} and share OTP {otp} to reverse immediately.",
        "Congratulations! You are selected for {bank} cashback reward of {amount}. Verify with OTP {otp}. Share only with our agent.",
        "{upi} refund of {amount} initiated. For instant credit, share the OTP sent to your number. Processing ID: REF{otp}",
        "URGENT: Someone is trying to transfer {amount} from your {bank}. To block this, share the OTP you just received. Call {phone}.",
        "Your {bank} loan EMI payment failed. Share OTP {otp} to set up auto-debit and avoid penalty charges of {amount}.",
        "RBI approved refund of {amount} for overcharged fees. Confirm OTP {otp} to receive in your {bank} account within 1 hour.",
    ],
    "link_phishing": [
        "You have won an Amazon gift card worth {amount}! Claim now before it expires: {link}",
        "Dear {name}, your {upi} reward points (worth {amount}) are expiring today. Redeem now: {link}",
        "Government subsidy of {amount} approved for your Aadhaar. Claim before March 31: {link}",
        "IRCTC: Your ticket refund of {amount} is ready. Process your refund here: {link}",
        "Flipkart Big Sale! You've been selected for exclusive discount of 90%. Shop now: {link}",
        "Your electricity bill payment of {amount} failed. Pay now to avoid disconnection: {link}",
        "{bank} important: New security update required. Login and verify: {link}. Failure to update within 48 hrs will suspend your account.",
        "PM Kisan Samman Nidhi: Your payment of {amount} is pending verification. Complete eKYC: {link}",
        "Jio/Airtel: Your mobile number will be disconnected in 24 hrs due to Aadhaar mismatch. Re-verify: {link}",
        "Income Tax Refund of {amount} for AY 2024-25 is pending. Submit bank details for direct deposit: {link}",
    ],
    "lottery_prize": [
        "CONGRATULATIONS {name}! You have won {amount} in Jio KBC Lucky Draw 2025. Claim ref: {otp}. Contact: {phone}",
        "WhatsApp lucky draw winner! You won {amount}. To claim, pay processing fee of ₹5,000. Contact {phone}. Ref: WA{otp}",
        "🎉 You are selected in PM Jan Dhan Yojana lottery! Prize: {amount}. Processing: {link}. Expires in 24 hours!",
        "Dear {name}, your mobile number won {amount} in Airtel Mega Lucky Draw. Claim immediately. Call {phone}.",
        "GOOGLE ANNUAL PRIZE: Congratulations! You have been randomly selected to receive {amount}. Claim at: {link}. Ref ID: GP{otp}",
        "Paytm Cashback Bonanza! You won {amount} cashback. Claim before midnight: {link}",
        "Dear Customer, you have won a Tata Nexon car + {amount} cash in HDFC Bank anniversary draw! Call {phone} with ref #{otp}",
        "Facebook lottery! Your profile was selected to win {amount}. Transfer fee ₹2,000 to receive. Details: {phone}",
        "🏆 IPL Season Lucky Winner! Prize: {amount} + VIP tickets. Confirm at {link}. Don't miss this!",
        "Amazon Prime Day random selection: You won iPhone 15 + {amount}. Claim: {link}. Ref: AMZ{otp}",
    ],
    "job_scam": [
        "HIRING: Earn ₹5,000-₹15,000/day from home. Simple typing/data entry work. No experience needed. WhatsApp: {phone}",
        "Amazon/Flipkart hiring for product review job. Earn {amount}/month. Join: {link}. Limited slots!",
        "Work from home opportunity! Earn ₹500-₹3000 per task. Investment: ₹1,000 only. Join now: {link}",
        "Dear {name}, you are selected for part-time job at Google India. Salary: {amount}/month. Register: {link}",
        "YouTube watching job! Watch 5 videos daily and earn ₹5000. No qualification needed. Apply: {link}",
        "Telegram task job: Complete simple tasks and earn daily income of ₹2000-₹8000. Start with ₹500 deposit. Join: {link}",
        "MNC hiring data entry operators. 2-3 hrs/day. Salary {amount}. Processing fee ₹999. Apply: {link}",
        "Government job vacancy! No exam required. Direct appointment. Salary {amount}. Registration: {link}. Limited time!",
        "Instagram influencer job: Post 2 reels daily, earn ₹10,000/week. Investment ₹2,000. WhatsApp: {phone}",
        "URGENT HIRING: {name}, your resume matched for senior role. Package {amount}/annum. Pay ₹3000 registration: {link}",
    ],
    "parcel_courier": [
        "Your {courier} parcel is held at customs. Contraband items detected. Pay clearance fine of {amount} or face legal action: {link}",
        "{courier} delivery failed: Address incomplete. Reschedule or parcel returns to sender. Update address: {link}. Tracking: {otp}",
        "ALERT: Your {courier} package contains restricted items. Customs has seized it. Pay penalty {amount} at {link} to release.",
        "Dear Customer, your international parcel from China is held by Indian Customs. Duty of {amount} pending. Pay: {link}",
        "{courier}: Your parcel #IND{otp} has arrived but delivery address is unverified. Confirm at {link} to avoid return.",
        "India Post: A parcel addressed to you contains suspicious items. Report to: {phone} or pay fine at {link}",
        "Your Amazon order is stuck at customs. Additional duty of {amount} required for delivery. Pay here: {link}",
        "{courier} ALERT: Drug enforcement has flagged your package. Contact {phone} immediately to avoid FIR under NDPS Act.",
        "A valuable package worth {amount} is waiting at {courier} hub. Pay ₹500 insurance fee for delivery: {link}",
        "Customs dept: International courier in your name intercepted. Contains foreign currency. Call {phone} or pay fine: {link}",
    ],
    "insurance_loan": [
        "Pre-approved personal loan of {amount} at 2% interest! No documents needed. Apply now: {link}. Limited period offer.",
        "Dear {name}, claim your LIC bonus of {amount}. Policy maturity payout ready. Submit bank details: {link}",
        "Instant loan approval! Get {amount} in 5 minutes. No CIBIL check. Minimal processing fee. Apply: {link}",
        "{bank} pre-approved loan for you! Amount: {amount} at lowest EMI. Disbursal in 2 hrs. Accept: {link}",
        "Your insurance claim of {amount} is approved! Pay ₹2,500 processing fee to receive payout. Details: {phone}",
        "Government Mudra Loan scheme: Get {amount} at 0% interest. No guarantor needed. Register: {link}",
        "CONGRATS! {bank} credit card limit increased to {amount}. Activate now: {link}. Offer valid 24 hours only.",
        "SBI Mudra Yojana: Loan of {amount} approved in your name. Pay ₹1,999 stamp duty for disbursement. {link}",
        "Your old LIC policy has unclaimed bonus of {amount}. Submit claim before it lapses: {link}. Ref: LIC{otp}",
        "Personal loan at ₹999/lakh EMI! Pre-approved {amount} for salaried individuals. 0 paperwork. Apply: {link}",
    ],
    "govt_impersonation": [
        "TRAI NOTICE: Your mobile number will be disconnected within 2 hours due to illegal activity reported. Press 1 or call {phone}.",
        "Income Tax Department: Refund of {amount} for AY 2024-25 pending. Verify your bank account: {link}. Expires today.",
        "EPFO Alert: Your PF withdrawal of {amount} requires Aadhaar verification. Complete at {link} within 24 hrs.",
        "Dear Citizen, your Aadhaar card is being used at 4 different locations. Block immediately: {link} or call {phone}.",
        "RBI circular: All bank accounts without PAN-Aadhaar link will be frozen. Complete linking: {link}. Last date extended.",
        "e-Court summons: Case filed against your Aadhaar #{otp}. Appear digitally: {link}. Non-appearance = warrant.",
        "Ministry of Finance: GST refund of {amount} credited. Verify bank details for release: {link}",
        "TRAI: Legal case #{otp} filed on your number for sending spam. Pay fine {amount} or number disconnected: {phone}",
        "Passport Office: Your passport application has discrepancy. Submit correction at {link} or application cancelled.",
        "Aadhaar Update: UIDAI requires biometric verification. Visit nearest center or update online: {link}. Ignore = Aadhaar deactivation.",
    ],
    "investment_scam": [
        "Earn {amount}/month guaranteed! Join our premium stock tips group. 100% accurate. WhatsApp: {phone}",
        "SEBI registered advisor: Triple your investment in 30 days. Minimum ₹10,000. Join: {link}",
        "Crypto trading opportunity: Invest ₹5,000 today, get {amount} in 7 days. Guaranteed by blockchain. Start: {link}",
        "Dear {name}, join our IPO allotment guaranteed scheme. Pay {amount} processing, get 10x returns. {phone}",
        "Forex trading job! We invest, you earn. Daily return of ₹2,000-₹10,000. Register: {link}. Zero risk!",
        "WhatsApp stock market group: Our calls gave 500% return last month. Join premium: {link}. Monthly fee ₹999 only.",
        "Binary options trading: Invest {amount} and withdraw {amount} profit daily. 100% automated. Start: {link}",
        "Mutual fund SIP with 50% annual guaranteed return. Better than FD. Invest: {link}. AMFI registered.",
        "Gold trading opportunity: Buy digital gold at 20% discount. Limited stock. Purchase: {link}",
        "NFT investment: Buy exclusive Indian art NFTs. Each ₹500 NFT worth {amount} in 6 months. Mint: {link}",
    ],
    "utility_scam": [
        "Your electricity will be disconnected today due to pending bill of {amount}. Pay immediately: {link}. -BSES/Tata Power",
        "URGENT: Gas connection will be terminated. Last date today. Outstanding: {amount}. Pay: {link}",
        "Water supply disconnection notice. Bill overdue: {amount}. Make payment to avoid disruption: {link}",
        "Dear consumer, your electricity meter reading shows tampering. Fine of {amount} levied. Pay at {link} or face FIR.",
        "MSEDCL/BESCOM: Your bill of {amount} is overdue. Supply will be cut in 3 hours. Quick pay: {link}",
    ],
}

# ── HAM Templates (7 categories, ~51 templates) ──────────────────────────────
HAM_TEMPLATES = {
    "bank_transactional": [
        "Your a/c XX5678 is credited with {amount} by NEFT. Avl bal: {amount}. -{bank}",
        "Dear Customer, {amount} debited from a/c XX1234 for UPI txn to {name}. Ref #{otp}. -{bank}",
        "OTP for {bank} net banking login is {otp}. Valid for 5 min. Do NOT share with anyone. -{bank}",
        "Your {bank} credit card payment of {amount} is received. Thank you. Outstanding: ₹0. -{bank}",
        "ATM withdrawal of {amount} from your {bank} a/c XX9876. If not done by you, call 1800XXXXXXX. -{bank}",
        "Your {bank} FD of {amount} has matured. Amount credited to savings a/c. -{bank}",
        "{bank}: EMI of {amount} for loan a/c XXXX456 debited successfully. Next EMI due: 5th April. -{bank}",
        "Auto-debit of {amount} processed for {bank} credit card. Avl limit: {amount}. -{bank}",
        "Your {bank} cheque #{otp} of {amount} has been cleared. -{bank}",
        "{bank}: IMPS transfer of {amount} to a/c ending 7890 successful. Ref: {otp}. -{bank}",
    ],
    "upi_transactional": [
        "{upi}: You paid {amount} to {name}. UPI Ref: {otp}. Balance: {amount}.",
        "{upi}: {amount} received from {name}. UPI Ref: {otp}.",
        "{upi}: Payment of {amount} to Swiggy successful. Order confirmed.",
        "{upi}: AutoPay of {amount} for Jio Recharge completed. Next due: 22 Mar.",
        "{upi}: Bill payment of {amount} to BESCOM electricity successful. Receipt: {otp}.",
        "Your {upi} UPI PIN has been changed successfully. If not done by you, call helpline immediately.",
        "{upi}: Cashback of ₹50 credited for your transaction of {amount}. Check app for details.",
    ],
    "ecommerce": [
        "Your Amazon order #402-{otp} has been shipped! Track: amazon.in/trackpackage. Delivery by 25 Feb.",
        "Flipkart: Your order for Samsung Galaxy is out for delivery today!",
        "Myntra: Your return request for order #{otp} has been processed. Refund of {amount} in 3-5 days.",
        "Zomato: Your order from Biryani Blues is being prepared. Delivery in 25-30 min. Track in app.",
        "Swiggy: Hurray! Use code SWIGGY50 for 50% off upto ₹100 on your next order. Valid till midnight.",
        "BigBasket: Your grocery delivery is scheduled for today 4-6 PM. Order #{otp}.",
        "Meesho: Flash sale! Extra 30% off on all fashion. Shop now on the Meesho app.",
    ],
    "government_legitimate": [
        "UIDAI: Your Aadhaar update request has been processed. New Aadhaar will be dispatched in 15 days.",
        "IRCTC: PNR {otp} - Train 12301 Rajdhani Exp, confirmed. Coach B2, Berth 45. Travel date: 28 Feb.",
        "DigiLocker: Your driving license has been successfully linked. Access anytime via digilocker.gov.in.",
        "CoWIN: Your COVID vaccination certificate is ready. Download from cowin.gov.in using ref #{otp}.",
        "e-Filing: ITR for AY 2025-26 processed. Refund of {amount} initiated to bank a/c ending 5678.",
        "UMANG: Your PF balance as on Jan 2026 is {amount}. For detailed statement, visit umang.gov.in.",
        "mParivahan: Your vehicle RC for MH02XX{otp} is due for renewal. Apply online at parivahan.gov.in.",
    ],
    "telecom": [
        "Jio: Your recharge of ₹399 is successful. Validity: 28 days. Data: 2GB/day. Enjoy!",
        "Airtel Thanks: Your prepaid plan of ₹599 is active. Unlimited calls + 2GB/day for 56 days.",
        "Vi: Data balance low. 150MB remaining. Recharge with ₹199 for 1.5GB/day. Dial *199# or visit Vi app.",
        "BSNL: Your broadband usage has crossed 80%. FUP limit: 100GB. Current: 82GB. -BSNL",
        "Jio: Your JioFiber bill of {amount} for Feb 2026 is generated. Pay by 5th March to avoid late fee.",
    ],
    "travel_booking": [
        "MakeMyTrip: Your flight booking PNR {otp} is confirmed. DEL→BOM on 15 Mar, 06:30 AM. Web check-in opens 48 hrs before.",
        "OYO: Booking confirmed! Check-in: 1 Mar, Hotel Grand Palace, Jaipur. Booking ID: {otp}.",
        "RedBus: Your bus ticket from Bangalore to Chennai on 28 Feb is confirmed. Seat: 2A, 3A. Boarding: Majestic.",
        "Ola: Your ride is arriving in 3 min. Driver: {name}, White Swift, KA01XX{otp}. Track in app.",
        "Uber: Your trip receipt. Distance: 12 km, Fare: {amount}. Payment via {upi}. Rate your driver!",
    ],
    "general_promotional": [
        "Wow! {bank} pre-approved personal loan at 10.5% p.a. Apply on net banking. T&C apply. -{bank}",
        "Domino's: Buy 1 Get 1 FREE on all pizzas! Use code BOGO. Order now on the app. T&C apply.",
        "Nykaa: End of season sale! Up to 50% off on top beauty brands. Shop now: nykaa.com",
        "Decathlon: Weekend special! Flat 20% off on all sports shoes. Visit your nearest store.",
        "CRED: Your CRED coins balance is 45,230. Redeem for exclusive rewards on the CRED app.",
        "Apollo 247: Your health check-up report is ready. View on the Apollo 247 app or visit apollopharmacy.in.",
        "Dear Customer, your {bank} locker rental of {amount} is due. Visit branch to renew. -{bank}",
    ],
}

# ── Generator Functions ───────────────────────────────────────────────────────
def fill_template(template):
    t = template
    t = t.replace("{bank}", random.choice(INDIAN_BANKS))
    t = t.replace("{upi}", random.choice(UPI_APPS))
    t = t.replace("{agency}", random.choice(GOVT_AGENCIES))
    t = t.replace("{courier}", random.choice(COURIER_SERVICES))
    t = t.replace("{link}", random.choice(FAKE_LINKS))
    t = t.replace("{amount}", random.choice(AMOUNTS))
    t = t.replace("{phone}", random.choice(PHONE_NUMBERS))
    t = t.replace("{otp}", random.choice(OTP_CODES))
    t = t.replace("{name}", random.choice(NAMES))
    return t

def add_noise(text):
    for _ in range(random.randint(0, 2)):
        op = random.choice(["case", "abbrev", "typo", "space"])
        if op == "case":
            r = random.random()
            if r < 0.15: text = text.upper()
            elif r < 0.3: text = text.lower()
        elif op == "abbrev":
            swaps = {"your": "ur", "please": "pls", "account": "a/c",
                     "immediately": "ASAP", "number": "no.", "message": "msg"}
            for old, new in swaps.items():
                if random.random() < 0.2:
                    text = text.replace(old, new, 1).replace(old.capitalize(), new.capitalize(), 1)
        elif op == "typo" and random.random() < 0.1 and len(text) > 5:
            i = random.randint(1, len(text) - 3)
            text = text[:i] + text[i+1] + text[i] + text[i+2:]
        elif op == "space" and random.random() < 0.08:
            i = random.randint(0, len(text) - 1)
            text = text[:i] + " " + text[i:]
    return text.strip()

# ── Generate All Synthetic Data (with DLT headers) ───────────────────────────
NUM_SCAM_PER_TEMPLATE = 30   # ~105 templates x 30 = ~3,150 scam
NUM_HAM_PER_TEMPLATE = 40    # ~51 templates x 40 = ~2,040 ham

synthetic_rows = []

# Generate SCAM data — each message gets a scam-style header
for category, templates in SCAM_TEMPLATES.items():
    for tmpl in templates:
        for _ in range(NUM_SCAM_PER_TEMPLATE):
            msg = add_noise(fill_template(tmpl))
            header = get_scam_header()
            text_with_header = format_with_header(header, msg)
            synthetic_rows.append({"text": text_with_header, "label": 1, "source": f"syn_{category}"})

# Generate HAM data — each message gets a proper DLT header
for category, templates in HAM_TEMPLATES.items():
    for tmpl in templates:
        for _ in range(NUM_HAM_PER_TEMPLATE):
            filled = fill_template(tmpl)
            msg = add_noise(filled)
            # Pick the right bank/upi from the filled template for header matching
            bank_used = next((b for b in INDIAN_BANKS if b in filled), None)
            upi_used = next((u for u in UPI_APPS if u in filled), None)
            courier_used = next((c for c in COURIER_SERVICES if c in filled), None)
            header = get_ham_header(category, bank=bank_used, upi=upi_used, courier=courier_used)
            text_with_header = format_with_header(header, msg)
            synthetic_rows.append({"text": text_with_header, "label": 0, "source": f"syn_{category}"})

df_synthetic = pd.DataFrame(synthetic_rows)
scam_n = df_synthetic["label"].sum()
print(f"Synthetic generated: {len(df_synthetic)} msgs (scam={scam_n}, ham={len(df_synthetic)-scam_n})")
print(f"\nSample SCAM (with header): {df_synthetic[df_synthetic['label']==1].iloc[0]['text'][:120]}...")
print(f"Sample HAM  (with header): {df_synthetic[df_synthetic['label']==0].iloc[0]['text'][:120]}...")

## 🔀 Cell 4 — Combine, Clean, Split & Tokenize

Merges all data sources, applies preprocessing, and creates PyTorch DataLoaders.

### Data Flow
```
Kaggle (UCI + India) ──┐
Synthetic Templates ───┼──→ Add Headers ──→ Dedupe ──→ 80/10/10 Split ──→ Tokenize
Personal SMS (upload) ─┘
```

### Processing Steps
1. **Add DLT headers** to Kaggle data (they don't have sender IDs)
2. **Upload `training_data.json`** — your personal SMS for real-world examples
3. **Deduplicate** exact matches
4. **Stratified split** — maintains scam/ham ratio across train/val/test
5. **Tokenize** with MobileBERT WordPiece (max_length=128)

### Input Format
```
"JD-SBINOT [SEP] Your a/c XX5678 is credited with Rs 25,000 by NEFT."
 ↑ Header        ↑ Message body
```

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4: COMBINE → ADD HEADERS TO KAGGLE DATA → CLEAN → SPLIT → TOKENIZE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Add synthetic headers to Kaggle datasets (they don't have DLT headers) ───
ALL_LEGIT_HEADERS = (LEGIT_GOVT_HEADERS + LEGIT_TELECOM_HEADERS + LEGIT_PROMO_HEADERS +
                     LEGIT_ECOM_HEADERS +
                     [h for hs in LEGIT_BANK_HEADERS.values() for h in hs] +
                     [h for hs in LEGIT_UPI_HEADERS.values() for h in hs])

def add_header_to_kaggle_row(row):
    """Add a synthetic DLT header to Kaggle dataset rows."""
    text = str(row["text"])
    if "[SEP]" in text:
        return text
    if row["label"] == 1:
        header = get_scam_header()
    else:
        header = None
        for bank, headers in LEGIT_BANK_HEADERS.items():
            if bank.lower() in text.lower():
                header = random.choice(headers)
                break
        if header is None:
            for app, headers in LEGIT_UPI_HEADERS.items():
                if app.lower() in text.lower():
                    header = random.choice(headers)
                    break
        if header is None:
            header = random.choice(ALL_LEGIT_HEADERS)
    return format_with_header(header, text)

print("Adding DLT headers to Kaggle datasets...")
if not df_uci.empty:
    df_uci["text"] = df_uci.apply(add_header_to_kaggle_row, axis=1)
    print(f"  UCI: headers added to {len(df_uci)} rows")
if not df_india.empty:
    df_india["text"] = df_india.apply(add_header_to_kaggle_row, axis=1)
    print(f"  India: headers added to {len(df_india)} rows")

# ── Load personal SMS data (training_data.json) ──────────────────────────────
df_personal = pd.DataFrame()
try:
    from google.colab import files
    print("\nUpload training_data.json (your personal SMS):")
    uploaded = files.upload()
    personal_file = list(uploaded.keys())[0]
except ImportError:
    personal_file = "training_data.json"  # local path

if Path(personal_file).exists():
    with open(personal_file, 'r', encoding='utf-8') as f:
        personal_data = json.load(f)
    msgs = personal_data.get('messages', personal_data if isinstance(personal_data, list) else [])

    personal_rows = []
    for m in msgs:
        text = m.get('text', '')
        addr = m.get('address', '')
        label = m.get('label', 0)
        # Format with header like other data
        formatted = format_with_header(addr, text) if addr else text
        personal_rows.append({'text': formatted, 'label': label, 'source': 'personal'})

    df_personal = pd.DataFrame(personal_rows)
    print(f"Personal SMS: {len(df_personal)} msgs (ham={len(df_personal)-df_personal['label'].sum()}, scam={df_personal['label'].sum()})")
else:
    print(f"No personal SMS file found ({personal_file}). Skipping.")

# ── Combine ───────────────────────────────────────────────────────────────────
dfs_to_concat = [df_uci, df_india, df_synthetic]
if not df_personal.empty:
    dfs_to_concat.append(df_personal)
df_all = pd.concat(dfs_to_concat, ignore_index=True)
df_all["text"] = df_all["text"].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip() if isinstance(x, str) else "")
df_all = df_all[df_all["text"].str.len() > 3].reset_index(drop=True)

before = len(df_all)
df_all = df_all.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Removed {before - len(df_all)} duplicates")
print(f"\nCOMBINED DATASET: {len(df_all)} msgs | scam={df_all['label'].sum()} | ham={len(df_all)-df_all['label'].sum()} | ratio={df_all['label'].mean():.1%}")
print(df_all.groupby("source")["label"].agg(["count","sum"]).rename(columns={"count":"total","sum":"scam"}).to_string())

print(f"\nSample rows with headers:")
for i in range(min(4, len(df_all))):
    print(f"  [{df_all.iloc[i]['label']}] {df_all.iloc[i]['text'][:100]}...")

df_all.to_csv(DATA_DIR / "combined_dataset.csv", index=False)

# ── Split ─────────────────────────────────────────────────────────────────────
X, y = df_all["text"].values, df_all["label"].values
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)
print(f"\nTrain={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

# ── Import torch & transformers here (deferred to avoid torchvision conflicts) ─
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import MobileBertTokenizer, MobileBertForSequenceClassification, get_linear_schedule_with_warmup

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# ── Tokenize ──────────────────────────────────────────────────────────────────
tokenizer = MobileBertTokenizer.from_pretrained("google/mobilebert-uncased")

class SMSDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "token_type_ids": self.encodings["token_type_ids"][idx],
            "labels": self.labels[idx],
        }

print("Tokenizing...")
train_dataset = SMSDataset(X_train, y_train)
val_dataset = SMSDataset(X_val, y_val)
test_dataset = SMSDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Done. Batches — train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

## 🧠 Cell 5 — Fine-tune MobileBERT (PyTorch)

Trains the classification head on top of frozen/unfrozen MobileBERT encoder.

### Training Configuration
| Parameter | Value |
|-----------|-------|
| Base Model | `google/mobilebert-uncased` |
| Optimizer | AdamW (lr=2e-5, eps=1e-8) |
| Scheduler | Linear warmup (10% of steps) |
| Batch Size | 32 |
| Max Epochs | 4 |
| Early Stopping | Patience=2 |
| Gradient Clipping | max_norm=1.0 |

### Output
- Classification report (precision, recall, F1)
- Confusion matrix
- Best checkpoint saved to `silver_guard_model/`

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5: LOAD → TRAIN → EVALUATE → SAVE MOBILEBERT (PYTORCH)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Load model ────────────────────────────────────────────────────────────────
model = MobileBertForSequenceClassification.from_pretrained("google/mobilebert-uncased", num_labels=2)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}, total_steps={total_steps}")

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_acc = 0.0
patience_counter = 0
PATIENCE = 2

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += len(batch["labels"])

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        if step % 50 == 0:
            print(f"  Epoch {epoch+1} step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    train_acc = correct / total
    avg_loss = total_loss / len(train_loader)

    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            val_correct += (preds == batch["labels"]).sum().item()
            val_total += len(batch["labels"])
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={avg_loss:.4f} train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        # Save best checkpoint
        model.save_pretrained("silver_guard_best")
        tokenizer.save_pretrained("silver_guard_best")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

# Load best checkpoint
print(f"\nLoading best model (val_acc={best_val_acc:.4f})...")
model = MobileBertForSequenceClassification.from_pretrained("silver_guard_best")
model.to(DEVICE)
model.eval()

# ── Evaluate on test set ──────────────────────────────────────────────────────
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

all_preds, all_labels = np.array(all_preds), np.array(all_labels)
print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=["Ham (Safe)", "Scam (Threat)"]))

cm = confusion_matrix(all_labels, all_preds)
print(f"Confusion Matrix:\n                Pred Ham  Pred Scam\nActual Ham   {cm[0][0]:>9}  {cm[0][1]:>9}\nActual Scam  {cm[1][0]:>9}  {cm[1][1]:>9}")
print(f"\nTest Accuracy: {(all_preds == all_labels).mean():.4f}")

# ── Save final model ─────────────────────────────────────────────────────────
MODEL_DIR = Path("silver_guard_model")
MODEL_DIR.mkdir(exist_ok=True)
model.save_pretrained(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print(f"Model saved to {MODEL_DIR}/")

## 📤 Cell 6 — Export to ONNX

Converts PyTorch model to ONNX format for mobile deployment via `onnxruntime_flutter`.

### Wrapper Architecture
```
Input: input_ids [1, 128], attention_mask [1, 128]
              ↓
       MobileBERT Encoder
              ↓
       Classification Head (2 logits)
              ↓
       Softmax → scam_probability
              ↓
Output: threat_score [1, 1]  (0.0 = safe, 1.0 = scam)
```

### Export Files
| File | Size | Purpose |
|------|------|---------|
| `silver_guard.onnx` | ~7 MB | Model weights |
| `vocab.txt` | ~230 KB | WordPiece vocabulary |
| `model_config.json` | <1 KB | Inference config |

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6: EXPORT PYTORCH → ONNX
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q onnx onnxruntime onnxscript
import onnx
import onnxruntime as ort

EXPORT_DIR = Path("silver_guard_export")
EXPORT_DIR.mkdir(exist_ok=True)
ONNX_PATH = EXPORT_DIR / "silver_guard.onnx"

# ── Wrapper: model → single threat score ──────────────────────────────────────
class SilverGuardWrapper(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.base_model.eval()

    def forward(self, input_ids, attention_mask):
        token_type_ids = torch.zeros_like(input_ids)
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        probs = F.softmax(outputs.logits, dim=-1)
        return probs[:, 1:2]  # [batch, 1] — scam probability

wrapper = SilverGuardWrapper(model)
wrapper.eval()
wrapper.to("cpu")

dummy_ids = torch.ones(1, MAX_LENGTH, dtype=torch.long)
dummy_mask = torch.ones(1, MAX_LENGTH, dtype=torch.long)
with torch.no_grad():
    test_score = wrapper(dummy_ids, dummy_mask)
print(f"Wrapper sanity check — threat score: {test_score.item():.4f}")

# ── Export to ONNX ────────────────────────────────────────────────────────────
print("Exporting to ONNX...")
torch.onnx.export(
    wrapper, (dummy_ids, dummy_mask), str(ONNX_PATH),
    opset_version=14,
    input_names=["input_ids", "attention_mask"],
    output_names=["threat_score"],
    dynamic_axes={"input_ids": {0: "batch"}, "attention_mask": {0: "batch"}, "threat_score": {0: "batch"}},
)

# ── If external data was created, convert to all-in-one ONNX ─────────────────
# PyTorch may split weights into silver_guard.onnx + silver_guard.onnx.data
# We merge them into a single self-contained file for easy deployment
DATA_SIDECAR = Path(str(ONNX_PATH) + ".data")
if DATA_SIDECAR.exists():
    print(f"External data detected ({DATA_SIDECAR.name}). Merging into single ONNX file...")
    onnx_model = onnx.load(str(ONNX_PATH), load_external_data=True)
    onnx.save_model(onnx_model, str(ONNX_PATH),
                    save_as_external_data=False)  # all weights inline
    DATA_SIDECAR.unlink()  # remove the now-unnecessary .data file
    print("Merged successfully — single self-contained ONNX file.")
else:
    onnx_model = onnx.load(str(ONNX_PATH))

onnx.checker.check_model(onnx_model)
print(f"ONNX saved: {ONNX_PATH} ({os.path.getsize(ONNX_PATH)/1024/1024:.1f} MB)")

# ── Verify with ONNX Runtime ─────────────────────────────────────────────────
session = ort.InferenceSession(str(ONNX_PATH))
ort_result = session.run(None, {"input_ids": dummy_ids.numpy(), "attention_mask": dummy_mask.numpy()})
print(f"ONNX Runtime verify — threat score: {ort_result[0][0][0]:.4f}")

# ── Save tokenizer files to export dir ────────────────────────────────────────
tokenizer.save_pretrained(str(EXPORT_DIR))

# Ensure vocab.txt exists (extract from tokenizer if needed)
vocab_path = EXPORT_DIR / "vocab.txt"
if not vocab_path.exists():
    print("vocab.txt not found, extracting from tokenizer vocabulary...")
    vocab = tokenizer.get_vocab()
    sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
    with open(vocab_path, "w", encoding="utf-8") as f:
        for token, _ in sorted_vocab:
            f.write(token + "\n")

# Save model config
with open(EXPORT_DIR / "model_config.json", "w") as f:
    json.dump({
        "max_length": MAX_LENGTH, "do_lower_case": True, "model_type": "mobilebert",
        "vocab_file": "vocab.txt", "model_file": "silver_guard.onnx",
        "labels": {"0": "ham", "1": "scam"},
        "input_names": ["input_ids", "attention_mask"], "output_name": "threat_score",
        "input_format": "HEADER [SEP] message_text",
    }, f, indent=2)

# Keep only the files needed for Flutter
KEEP_FILES = {"silver_guard.onnx", "vocab.txt", "model_config.json"}
for f in list(EXPORT_DIR.iterdir()):
    if f.is_file() and f.name not in KEEP_FILES:
        f.unlink()

print(f"\nFiles ready for Flutter:")
for p in sorted(EXPORT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:30s} {os.path.getsize(p)/1024:.1f} KB")

## 🧪 Cell 7 — Test ONNX Inference & Download

Validates the exported model on hand-picked test cases and packages for Flutter.

### Test Categories (20 cases)
| Category | Expected | Tests |
|----------|----------|-------|
| Scam + Phone Number | 🚨 SCAM | CBI impersonation, bank freeze, OTP fraud |
| Scam + Spoofed Header | 🚨 SCAM | VM-URGENT, XX-WINNER fake IDs |
| Ham + Bank DLT | ✅ SAFE | Credit/debit alerts from JD-SBINOT |
| Ham + UPI DLT | ✅ SAFE | PhonePe, GPay transaction confirmations |
| Ham + Govt DLT | ✅ SAFE | IRCTC, UIDAI legitimate messages |
| Ham + No Header | ✅ SAFE | Personal messages from contacts |
| Mismatch (Govt claim + phone) | 🚨 SCAM | Income Tax from +91 number |

### Download
Creates `silver_guard_flutter.zip` containing all files needed for Flutter integration.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7: TEST ONNX INFERENCE & DOWNLOAD
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def predict_threat(header, message):
    combined = f"{header} [SEP] {message}" if header else message
    enc = tokenizer(combined, max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="np")
    result = session.run(None, {
        "input_ids": enc["input_ids"].astype(np.int64),
        "attention_mask": enc["attention_mask"].astype(np.int64),
    })
    return float(result[0][0][0])

tests = [
    ("Scam+PhoneNum",   "+919876543210",  "This is CBI. Your Aadhaar is linked to a money laundering case worth 50 lakh. A digital arrest warrant is issued. Call immediately."),
    ("Scam+PhoneNum",   "+917654321098",  "ALERT: Your SBI account has been FROZEN due to suspicious activity. Update KYC immediately at bit.ly/sbi-kyc or account will be closed."),
    ("Scam+PhoneNum",   "+918765432109",  "Your HDFC Bank refund of Rs 15,000 is pending. Share OTP 847293 with our executive to process the refund."),
    ("Scam+Spoofed",    "VM-URGENT",      "TRAI NOTICE: Your mobile number will be disconnected within 2 hours due to illegal activity. Press 1 or call now."),
    ("Scam+Spoofed",    "XX-WINNER",      "CONGRATULATIONS! You have won Rs 25 lakh in Jio KBC Lucky Draw 2025. Call to claim. Ref: KBC892341"),
    ("Scam+FakeBank",   "JD-SBINOP",      "Dear Customer, your SBI account has been FROZEN. Click bit.ly/sbi-verify to update KYC or account will be closed permanently."),
    ("Scam+FakeGovt",   "BZ-HDFCFR",      "HIRING: Earn Rs 5000-15000 daily from home! Simple data entry work. No experience needed. WhatsApp now."),
    ("Scam+PhoneNum",   "+916543210987",  "Earn 5 lakh per month guaranteed! Join our premium stock tips group. 100% accurate calls."),
    ("Ham+BankDLT",     "JD-SBINOT",      "Your a/c XX5678 is credited with Rs 25,000 by NEFT. Avl bal: Rs 1,45,230. -SBI"),
    ("Ham+UPIDLT",      "JD-PHONPE",      "PhonePe: You paid Rs 450 to Swiggy. UPI Ref: 304827156482. Balance: Rs 12,350."),
    ("Ham+BankOTP",     "AD-HDFCBK",      "Your OTP for HDFC net banking login is 482901. Valid for 5 min. Do NOT share with anyone. -HDFC Bank"),
    ("Ham+EcomDLT",     "JD-AMZNIN",      "Your Amazon order #402-7839201 has been shipped! Track: amazon.in/trackpackage. Delivery by 25 Feb."),
    ("Ham+GovtDLT",     "JD-IRCTCG",      "IRCTC: PNR 4523891012 - Train 12301 Rajdhani Exp, confirmed. Coach B2, Berth 45. Travel date: 28 Feb."),
    ("Ham+GovtDLT",     "DL-UIDAIG",      "UIDAI: Your Aadhaar update request has been processed. New Aadhaar will be dispatched in 15 days."),
    ("Ham+PromoDLT",    "DM-DOMINO",      "Domino's: Buy 1 Get 1 FREE on all pizzas! Use code BOGO. Order now on the app. T&C apply."),
    ("Ham+TelecomDLT",  "JD-JIOINF",      "Jio: Your recharge of Rs 399 is successful. Validity: 28 days. Data: 2GB/day. Enjoy!"),
    ("Ham+NoHeader",    "",                "Hey bro, are we meeting for dinner tonight? Let me know the time and place."),
    ("Ham+NoHeader",    "",                "Mom said to pick up milk and bread on your way home. Also get some onions."),
    ("MismatchGovt",    "+919999888877",  "This is Income Tax Department. Your refund of Rs 50000 is approved. Click here to verify your bank details."),
    ("MismatchBank",    "QQ-REWARD",      "Your SBI account KYC has expired. Update now at secure-bankupd.com/kyc or account will be closed."),
]

print(f"\n{'='*100}")
print(f"{'TYPE':<18} {'HEADER':<18} {'SCORE':>6}  {'VERDICT':<10}  MESSAGE")
print(f"{'='*100}")
for t, hdr, msg in tests:
    s = predict_threat(hdr, msg)
    v = "🚨 SCAM" if s > 0.5 else "✅ SAFE"
    print(f"{t:<18} {hdr:<18} {s:>5.3f}  {v:<10}  {msg[:45]}...")
print(f"{'='*100}")

# ── Zip & Download ────────────────────────────────────────────────────────────
shutil.make_archive("silver_guard_flutter", "zip", str(EXPORT_DIR))
print(f"\nCreated silver_guard_flutter.zip")

try:
    from google.colab import files
    files.download("silver_guard_flutter.zip")
    print("Download started!")
except ImportError:
    print(f"Not on Colab. Zip at: {os.path.abspath('silver_guard_flutter.zip')}")

print(f"""
FLUTTER INTEGRATION:
  1. Unzip into assets/ (silver_guard.onnx, vocab.txt, model_config.json)
  2. pubspec.yaml:
       flutter:
         assets:
           - assets/silver_guard.onnx
           - assets/vocab.txt
           - assets/model_config.json
  3. Add dependency: onnxruntime_flutter: ^1.0.0  (or latest)
  4. Dart usage:
       final session = await OrtSession.fromAsset('assets/silver_guard.onnx');
       // Combine header + message: "$header [SEP] $message"
       // Tokenize with WordPiece, pad to 128, run session
  5. If no header available (personal SMS), just pass the message body alone.
""")

## 💾 Cell 8 — Backup Full Content Folder

Downloads the entire `/content` directory as a backup zip (includes model checkpoints, datasets, exports).

In [ ]:
import os
from google.colab import files

# 1. Define the name of your zip file
zip_name = '/content/silver_guard_backup.zip'

# 2. Zip the /content directory via command line.
# CRITICAL: We use '-x' to exclude your Google Drive, the default sample_data,
# and the zip file itself so it doesn't get stuck in an infinite loop.
!zip -r {zip_name} /content -x "/content/drive/*" -x "/content/sample_data/*" -x "{zip_name}"

# 3. Trigger the browser download
print(f"Preparing to download {zip_name}...")
files.download(zip_name)

## 📱 Cell 9 — Evaluate Personal SMS Messages

Tests the model on **your real phone messages** (exported from Phone Link app as JSON).

### Input Format
Expects `sms_export.json` with structure:
```json
{
  "messages": [
    {"text": "...", "address": "JD-SBINOT", "kind": "received"},
    ...
  ]
}
```

### Output
- Score distribution histogram
- False positives (legit messages flagged as scam)
- Borderline cases (score 0.3–0.5)
- Downloadable `personal_sms_scored.csv`

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8: EVALUATE PERSONAL SMS MESSAGES (JSON from Phone Link export)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Upload file ───────────────────────────────────────────────────────────────
try:
    from google.colab import files
    print("Upload your SMS export (.json or .csv):")
    uploaded = files.upload()
    upload_name = list(uploaded.keys())[0]
except ImportError:
    upload_name = "sms_export.json"  # change this if running locally
    print(f"Not on Colab. Reading from: {upload_name}")

# ── Load into DataFrame ──────────────────────────────────────────────────────
if upload_name.endswith(".json"):
    with open(upload_name, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    # Handle nested format: { "messages": [...] }
    if isinstance(raw_data, dict) and "messages" in raw_data:
        df_phone = pd.DataFrame(raw_data["messages"])
        print(f"JSON format: nested (version={raw_data.get('version','?')}, total={raw_data.get('total_messages','?')})")
    elif isinstance(raw_data, list):
        df_phone = pd.DataFrame(raw_data)
        print("JSON format: flat array")
    else:
        raise ValueError("Unrecognized JSON structure. Expected {messages: [...]} or [...]")
elif upload_name.endswith(".csv"):
    df_phone = pd.read_csv(upload_name)
else:
    raise ValueError(f"Unsupported file type: {upload_name}. Use .json or .csv")

print(f"Loaded {len(df_phone)} messages | Columns: {list(df_phone.columns)}")

# ── Filter: only received messages (skip sent) ───────────────────────────────
if "kind" in df_phone.columns:
    sent_count = (df_phone["kind"] == "sent").sum()
    df_phone = df_phone[df_phone["kind"] != "sent"].reset_index(drop=True)
    print(f"Filtered out {sent_count} sent messages -> {len(df_phone)} received messages")

# ── Identify columns ─────────────────────────────────────────────────────────
msg_col = "text"      # your export uses "text"
sender_col = "address" # your export uses "address" (DLT header or phone number)
print(f"Message column: '{msg_col}' | Sender column: '{sender_col}'")

# ── Clean ─────────────────────────────────────────────────────────────────────
df_phone[msg_col] = df_phone[msg_col].astype(str).str.strip()
df_phone = df_phone[df_phone[msg_col].str.len() > 3].reset_index(drop=True)
print(f"After filtering short/empty: {len(df_phone)} messages")

# ── Run inference ─────────────────────────────────────────────────────────────
print(f"\nScoring {len(df_phone)} messages...")
scores = []
for idx in range(len(df_phone)):
    msg = df_phone[msg_col].iloc[idx]
    header = str(df_phone[sender_col].iloc[idx]).strip() if sender_col else ""
    score = predict_threat(header, msg)
    scores.append(score)
    if (idx + 1) % 500 == 0:
        print(f"  {idx + 1}/{len(df_phone)} done...")

df_phone["threat_score"] = scores
df_phone["verdict"] = df_phone["threat_score"].apply(lambda s: "SCAM" if s > 0.5 else "SAFE")
print(f"Done! All {len(df_phone)} messages scored.\n")

# ── Summary ───────────────────────────────────────────────────────────────────
n_total = len(df_phone)
n_scam = (df_phone["verdict"] == "SCAM").sum()
n_safe = (df_phone["verdict"] == "SAFE").sum()

print("=" * 70)
print("PERSONAL SMS EVALUATION SUMMARY")
print("=" * 70)
print(f"Total messages:     {n_total}")
print(f"Classified SAFE:    {n_safe}  ({n_safe/n_total*100:.1f}%)")
print(f"Classified SCAM:    {n_scam}  ({n_scam/n_total*100:.1f}%)")
print(f"Avg threat score:   {df_phone['threat_score'].mean():.4f}")
print(f"Score range:        {df_phone['threat_score'].min():.4f} - {df_phone['threat_score'].max():.4f}")
print("=" * 70)

# ── Score distribution ────────────────────────────────────────────────────────
bins = [(0, 0.1), (0.1, 0.2), (0.2, 0.3), (0.3, 0.4), (0.4, 0.5),
        (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
print("\nScore Distribution:")
for lo, hi in bins:
    count = ((df_phone["threat_score"] >= lo) & (df_phone["threat_score"] < hi)).sum()
    bar = "█" * (count * 40 // max(n_total, 1))
    marker = " <-- SCAM ZONE" if lo >= 0.5 else ""
    print(f"  {lo:.1f}-{hi:.1f}: {count:>5}  {bar}{marker}")

# ── False positives (flagged as scam but all your msgs are legit) ─────────────
if n_scam > 0:
    print(f"\n{'='*70}")
    print(f"FALSE POSITIVES - {n_scam} legit messages the model thinks are scam:")
    print(f"{'='*70}")
    fp = df_phone[df_phone["verdict"] == "SCAM"].sort_values("threat_score", ascending=False)
    for i, (_, row) in enumerate(fp.iterrows()):
        sender = f" [{row[sender_col]}]" if sender_col else ""
        print(f"\n  #{i+1} (score: {row['threat_score']:.3f}){sender}")
        print(f"  {row[msg_col][:200]}")
else:
    print("\nNo false positives - all personal messages classified as SAFE.")

# ── Borderline (0.3-0.5) ─────────────────────────────────────────────────────
borderline = df_phone[(df_phone["threat_score"] >= 0.3) & (df_phone["threat_score"] < 0.5)]
if len(borderline) > 0:
    print(f"\n{'='*70}")
    print(f"BORDERLINE - {len(borderline)} messages scored 0.3-0.5 (not flagged but notable):")
    print(f"{'='*70}")
    bl = borderline.sort_values("threat_score", ascending=False).head(20)
    for i, (_, row) in enumerate(bl.iterrows()):
        sender = f" [{row[sender_col]}]" if sender_col else ""
        print(f"  #{i+1} ({row['threat_score']:.3f}){sender}  {row[msg_col][:120]}")
    if len(borderline) > 20:
        print(f"  ... +{len(borderline) - 20} more")

# ── Save scored CSV ───────────────────────────────────────────────────────────
output_path = "personal_sms_scored.csv"
df_phone.to_csv(output_path, index=False)
print(f"\nScored results saved to: {output_path}")
try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    pass